# Ablation

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 16.11.2025
- Date of Last Modification: 16.11.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries 

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
sys.path.append("../../utils")

In [145]:
import os
import pickle
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.special import softmax
from datasets import concatenate_datasets, load_from_disk
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split

from app.infer import (embed_dataset,
                       get_average_gene_embed,
                       get_gene_embed,
                       harmonize_adata,
                       perturb_dataset,
                       tokenize_adata)
from nichejepa.evaluation import (compute_neighborhood_composition,
                                  get_adata_cellphonedb_lr_pairs,
                                  get_adata_omnipath_lr_pairs,
                                  knn_classifier,
                                  linear_classifier,
                                  linear_regressor,
                                  plot_roc_curve)
from nichejepa.preprocessors.filters import filter_cells
from nichejepa.preprocessors.graph import construct_neighbor_graph

from analysis_utils import *

## 1.2 Define Params

In [64]:
model_timestamp = '11062025_183320_055'

# Inference params
emb_layer = None
top_k = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_tokens = None
force_recompute_output_embed = False

# Tokenizer params
num_shards = 32
nproc = 4
n_neighs = 10
force_retokenize = False

## 1.3 Configure Paths

In [6]:
artifacts_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'
model_folder_path = f'{artifacts_folder_path}/models/{model_timestamp}'

In [7]:
os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/results', exist_ok=True)

## 1.4 Run Notebook Setup

In [8]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [9]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [10]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [11]:
sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

# 2. Experiments

## 2.1 xhs1000-39b_1p (Xenium Human Skin Eczema)

In [106]:
# Dataset variables
dataset_name = 'xhs1000-39b_1p'
cell_type_key = 'annotation_new7'
niche_key = 'NICHE_NAMES'
sample_key = 'sample'
batch_key = 'batch'
file_paths = [
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/adata_batch11.h5ad', # held out from pretraining
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/adata_batch19.h5ad', # held out from pretraining
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/adata_batch32.h5ad', # held out from pretraining
]
spot_size = 20

### 2.1.1 Visualize Ground Truth Niches

In [107]:
adata_list = []

for file_path in file_paths:
    # Load AnnData
    adata = sc.read_h5ad(file_path)

    # Harmonize AnnData
    adata = harmonize_adata(adata)
    adata.obs[batch_key] = adata.uns[batch_key]
    adata_list.append(adata)

STEP 1: ADDING ENSEMBL IDS...
Adding ensembl IDs from release 110...


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Number of genes with matching ensembl IDs:           4948.
Number of genes skipped:           0.
STEP 2: ADDING SPECIAL VALUES...
STEP 1: ADDING ENSEMBL IDS...
Adding ensembl IDs from release 110...


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Number of genes with matching ensembl IDs:           4948.
Number of genes skipped:           0.
STEP 2: ADDING SPECIAL VALUES...


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


STEP 1: ADDING ENSEMBL IDS...
Adding ensembl IDs from release 110...


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Number of genes with matching ensembl IDs:           4948.
Number of genes skipped:           0.
STEP 2: ADDING SPECIAL VALUES...


In [108]:
set(adata_list[0].obs[niche_key].unique().tolist() +
    adata_list[1].obs[niche_key].unique().tolist() +
    adata_list[2].obs[niche_key].unique().tolist())

{'Basal epidermis/superficial dermis',
 'Epidermis with T2',
 'Interstitial stroma',
 'KC',
 'KC_T2_LC',
 'Muscle',
 'Perivascular depeer',
 'Perivascular superficial',
 'Schwann_KC_HF',
 'Sebocyte',
 'Sweat gland1',
 'Sweat gland2'}

In [ ]:
# Define niche palette
label2niche = {
    '0': 'Basal epidermis/superficial dermis',
    '1': 'Perivascular superficial',
    '2': 'KC_T2_LC',
    '3': 'KC',
    '4': 'Perivascular depeer',
    '5': 'Interstitial stroma',
    '6': 'Sebocyte',
    '7': 'Schwann_KC_HF',
    '8': 'Muscle',
    '9': 'Sweat gland2',
    '10': 'Sweat gland1',
    '11': 'Epidermis with T2'
}

niche_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

In [ ]:
# Visualize ground truth niche labels in training samples
sample = list(set(adata_list[0].obs[sample_key].tolist()))[0]
batch = list(set(adata_list[0].obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[0],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)

sample = list(set(adata_list[2].obs[sample_key].tolist()))[0]
batch = list(set(adata_list[2].obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[2],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)

In [ ]:
# Visualize ground truth niche labels in test sample
sample = list(set(adata_list[1].obs[sample_key].tolist()))[0]
batch = list(set(adata_list[1].obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[1],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)

In [ ]:
# Define cell type palette
cell_type_palette = {
    'KC4-5': '#1f78b4',
    'KC3': '#33a02c',
    'KC1': '#fb9a99',
    'Melanocyte': '#e31a1c',
    'KC2': '#a6cee3',
    'F2: Universal': '#fdbf6f',
    'Mac': '#b2df8a',
    'cDC2': '#cab2d6',
    'T': '#6a3d9a',
    'F3: TRC-like': '#ff7f00',
    'F2/3: Bridge': '#b15928',
    'T_GZMK+': '#ffff99',
    'F1: Superficial': '#8dd3c7',
    'VE/Pericyte': '#ffffb3',
    'cDC1': '#bebada',
    'LE': '#fb8072',
    'F5: Schwann-like': '#80b1d3',
    'Sweat gland channel outer': '#fdb462',
    'Sweat gland': '#b3de69',
    'F4: Mixed': '#fccde5',
    'Treg': '#d9d9d9',
    'Nonspecific': '#bc80bd',
    'Pre-adipocyte PPARG+ANGPTL8': '#ccebc5',
    'MigDC': '#ffed6f',
    'LC': '#a1dab4',
    'T_ITGAEhi_IL22': '#41b6c4',
    'T_vascular': '#2c7fb8',
    'Mast cell': '#253494',
    'Muscle': '#fed976',
    'KC_HF_Basal': '#e31a1c',
    'KC_HF_ORS': '#bd0026',
    'Sebocyte_CCL19': '#54278f',
    'Schwann': '#756bb1',
    'Sebocyte_PPARG': '#9e9ac8',
    'ILC2/Tnaive': '#6baed6',
    'F4: TNN+COCH+': '#4292c6',
    'pDC': '#2171b5',
    'KC_HF_inner': '#08519c',
    'LC/MigDC': '#08306b',
    'Plasma cell': '#ff66a1'
}

In [ ]:
# Visualize ground truth cell type labels in training samples
sample = list(set(adata_list[0].obs[sample_key].tolist()))[0]
batch = list(set(adata_list[0].obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_cell_type_label_transfer_ground_truth_cell_types.svg'
sc.pl.spatial(
    adata_list[0],
    color=cell_type_key,
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)

sample = list(set(adata_list[2].obs[sample_key].tolist()))[0]
batch = list(set(adata_list[2].obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_cell_type_label_transfer_ground_truth_cell_types.svg'
sc.pl.spatial(
    adata_list[2],
    color=cell_type_key,
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)  

In [ ]:
# Visualize ground truth cell type labels in test sample
sample = list(set(adata_list[1].obs[sample_key].tolist()))[0]
batch = list(set(adata_list[1].obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_cell_type_label_transfer_ground_truth_cell_types.svg'
sc.pl.spatial(
    adata_list[1],
    color=cell_type_key,
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)

### 2.1.2 NicheJEPA

In [111]:
method = 'NicheJEPA'
niche_key = 'niche_type'
emb_key = 'neighborhood_emb'

#### 2.1.2.1 Retrieve Embeddings

In [112]:
# Cell and neighborhood embeddings
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded_{model_timestamp}.h5ad')

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [62]:
# Gene embeddings
dataset_list = []

for file_path in file_paths:
    # Load tokenized AnnData or tokenize data
    dataset_path = f'{artifacts_folder_path}/{dataset_name}/{file_path.split("/")[-1].split(".")[0]}.dataset'
    if os.path.exists(dataset_path) and not force_retokenize:
        dataset = load_from_disk(dataset_path)
    else:
        # Tokenize AnnData
        dataset = tokenize_adata(
            adata,
            model_folder_path,
            nproc=nproc,
            processing_mode='parallel')

        # Save tokenized data
        os.makedirs(f'{artifacts_folder_path}/{dataset_name}', exist_ok=True)
        dataset.save_to_disk(
            dataset_path,
            num_shards=num_shards)

    dataset_list.append(dataset)

full_dataset = concatenate_datasets(dataset_list)

#### 2.1.2.2 Train Niche Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

##### 2.1.2.2.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_{model_timestamp}_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.2.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate knn niche transfer classifier
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_{model_timestamp}_niche_label_transfer_knn_classifer_predicted_niches.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.2.3 Train Cell Type Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

##### 2.1.2.3.1 Linear Classifier

In [ ]:
# Retrieve features and labels
cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm['cell_emb']

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs['sample'] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs['sample'] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    cell_type_features_train = cell_type_features[train_indices]
    cell_type_features_val = cell_type_features[val_indices]
    cell_type_features_test = cell_type_features[test_indices]
    cell_type_labels_train = cell_type_labels[train_indices]
    cell_type_labels_val = cell_type_labels[val_indices]
    cell_type_labels_test = cell_type_labels[test_indices]

    # Train and evaluate linear cell type transfer classifier
    pred_cell_type_labels_test, _, _, model = linear_classifier(
        features_train=cell_type_features_train,
        labels_train=cell_type_labels_train,
        features_val=cell_type_features_val,
        labels_val=cell_type_labels_val,
        features_test=cell_type_features_test,
        labels_test=cell_type_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_cell_type_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )

    #print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted cell type labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_{model_timestamp}_cell_type_label_transfer_linear_classifer_predicted_cell_types_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_linear',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.2.3.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate knn cell type transfer classifier
pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_cell_type_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_{model_timestamp}_cell_type_label_transfer_knn_classifer_predicted_cell_types.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.2.4 Train Neighborhood Composition Classifiers

In [ ]:
# Compute spatial neighborhood graph
adata_batch_list = []
batches = adata_emb.obs[batch_key].unique().tolist()

for batch in batches:
    adata_batch = adata_emb[adata_emb.obs[batch_key] == batch]
    print('Computing spatial neighborhood...')
    # Construct neighbor graph
    adata_batch = construct_neighbor_graph(
        adata_batch,
        n_neighs=n_neighs,
        radius=None,
        delaunay=None,
        include_self_loop=False,
        compute_neighbor_counts=False)
    adata_batch_list.append(adata_batch)

adata_combined = sc.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
connectivities = sp.vstack(batch_connectivities)
adata_combined.obsp['spatial_connectivities'] = connectivities

composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata_emb.obsm['neighborhood_emb']

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get neighborhood composition labels and features
    composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
    cell_type_composition_labels = composition_df.values
    niche_features = adata_emb.obsm['neighborhood_emb']
    
    # Get train and validation indices
    train_val_indices = (adata_emb.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_composition_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        #stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_emb.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
    cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
    cell_type_composition_labels_test = cell_type_composition_labels[test_indices]
    
    pred_cell_type_composition_labels, model = linear_regressor(
        features_train=niche_features_train,
        labels_train=cell_type_composition_labels_train,
        features_val=niche_features_val,
        labels_val=cell_type_composition_labels_val,
        features_test=niche_features_test,
        labels_test=cell_type_composition_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_neighborhood_composition_linear_classifier_cls_report_seed_{seed}.txt'
    )
    
    print(pred_cell_type_composition_labels)

#### 2.1.2.5 Ligand-Receptor Prediction

##### 2.1.2.5.1 CellPhoneDB

In [141]:
lr_database = 'cellphonedb'

In [142]:
# Get Ensembl IDs
adata_emb.var = adata_list[0].var

In [143]:
# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_emb,
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_emb,
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

print(lr_interaction_df)

Downloaded cellphonedb.zip into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded complex_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded gene_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded interaction_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded protein_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded uniprot_synonyms.tsv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0/sources
Downloaded transcription_factor_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0/sources

              ligand         receptor
0    ENSG00000112715  ENSG00000128052
1    ENSG00000111144  ENSG00000144230
2    ENSG00000141433  ENSG00000114812
3    ENSG00000226979  ENSG00000028137
4    ENSG00000241563  ENSG000

In [ ]:
# Retrieve cell gene embeddings for receptors and neighborhood gene embeddings for ligands
file_path = f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_gene_embed_{lr_database}.pkl'

if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        output_gene_embed = pickle.load(f)
else:
    cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
    neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()
    
    output_gene_embed = get_gene_embed(
        dataset=full_dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        cell_gene_ensembl_id=cell_gene_ensembl_id,
        neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
    )
    
    with open(file_path, "wb") as f:
        pickle.dump(output_gene_embed, f) # shape (414, 384)

STEP 1: LOADING CONFIG...
STEP 2: GETTING GENE EMBEDDINGS


INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (seg_embed): Embedding(12, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr', 'iter_number']


0it [00:00, ?it/s]/nfs/team361/sb75/nichejepa/src/app/infer.py:1522: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


In [120]:
# Retrieve cell gene embeddings for receptors and neighborhood gene embeddings for ligands
file_path = f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_average_gene_embed_{lr_database}.pkl'

if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        output_average_gene_embed = pickle.load(f)
else:
    cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
    neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()
    
    output_average_gene_embed = get_average_gene_embed(
        dataset=full_dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        cell_gene_ensembl_id=cell_gene_ensembl_id,
        neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
    )
    
    with open(file_path, "wb") as f:
        pickle.dump(output_average_gene_embed, f) # shape (414, 384)

In [139]:
# Filter embeddings with too few occurrences in either neighborhood or cell gene
min_count_threshhold = 10

neighborhood_gene_emb_avg = output_average_gene_embed['neighborhood_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]
cell_gene_emb_avg = output_average_gene_embed['cell_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]

# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(187, 768)
(187, 768)


In [140]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 13.4335, Val Loss = 0.6900
Epoch 2: Train Loss = 13.2664, Val Loss = 0.6986
Epoch 3: Train Loss = 13.0901, Val Loss = 0.7009
Epoch 4: Train Loss = 12.9321, Val Loss = 0.7089
Epoch 5: Train Loss = 12.8099, Val Loss = 0.7142
Epoch 6: Train Loss = 12.7356, Val Loss = 0.7181
Epoch 7: Train Loss = 12.6468, Val Loss = 0.7207
Epoch 8: Train Loss = 12.4682, Val Loss = 0.7260
Epoch 9: Train Loss = 12.4077, Val Loss = 0.7340
Epoch 10: Train Loss = 12.2872, Val Loss = 0.7326
Epoch 11: Train Loss = 12.2000, Val Loss = 0.7395
Early stopping at epoch 11

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.5357    0.7895    0.6383        19
           1     0.6000    0.3158    0.4138        19

    accuracy                         0.5526        38
   macro avg     0.5679    0.5526    0.5260        38
weighted avg     0.5679    0.5526    0.5260        38


--- Evaluation Report saved. ---
Using device:

##### 2.1.2.5.2 OmniPath

In [78]:
lr_database = 'omnipath'

In [79]:
# Get Ensembl IDs
adata_emb.var = adata_list[0].var

In [89]:
# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_emb,
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_emb,
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

                ligand         receptor
66     ENSG00000187266  ENSG00000138741
84     ENSG00000010810  ENSG00000133107
111    ENSG00000010810  ENSG00000137672
183    ENSG00000010810  ENSG00000111199
337    ENSG00000133116  ENSG00000165125
...                ...              ...
88693  ENSG00000113520  ENSG00000166888
89474  ENSG00000067182  ENSG00000065559
90565  ENSG00000019991  ENSG00000100030
90821  ENSG00000125538  ENSG00000105397
93513  ENSG00000136244  ENSG00000168610

[1159 rows x 2 columns]


In [88]:
lr_interaction_df

,ligand,receptor
66,ENSG00000187266,ENSG00000138741
84,ENSG00000010810,ENSG00000133107
111,ENSG00000010810,ENSG00000137672
183,ENSG00000010810,ENSG00000111199
337,ENSG00000133116,ENSG00000165125
...,...,...
88693,ENSG00000113520,ENSG00000166888
89474,ENSG00000067182,ENSG00000065559
90565,ENSG00000019991,ENSG00000100030
90821,ENSG00000125538,ENSG00000105397


In [92]:
# Retrieve cell gene embeddings for receptors and neighborhood gene embeddings for ligands
file_path = f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_average_gene_embed_{lr_database}.pkl'

if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        output_average_gene_embed = pickle.load(f)
else:
    cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
    neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()
    
    output_average_gene_embed = get_average_gene_embed(
        dataset=full_dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        cell_gene_ensembl_id=cell_gene_ensembl_id,
        neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
    )
    
    with open(file_path, "wb") as f:
        pickle.dump(output_average_gene_embed, f) # shape (1159, 384)

STEP 1: LOADING CONFIG...
STEP 2: GETTING AVERAGE GENE EMBEDDINGS


INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCombinedEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (seg_embed): Embedding(12, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr', 'iter_number']


0it [00:00, ?it/s]/nfs/team361/sb75/nichejepa/src/app/infer.py:1522: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(
/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
1677it [2:24:41,  5.18s/it]
/nfs/team361/sb75/nichejepa/src/app/infer.py:1897: RuntimeWarning: invalid value encountered in divide
  cell_gene_emb_features.append((sum_emb / count_emb).reshape(1, -1))
/nfs/team361/sb75/nichejepa/src/app/infer.py:1902: RuntimeWarning: invalid value encountered in divide
  spatial_cell_gene_emb_features.append((spatial_sum_emb / spatial_count_emb).reshape(1, -1))
/nfs/team361/sb75/nichejepa/src/app/infer.py:190

In [94]:
# Filter embeddings with too few occurrences in either neighborhood or cell gene
min_count_threshhold = 10

neighborhood_gene_emb_avg = output_average_gene_embed['neighborhood_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]
cell_gene_emb_avg = output_average_gene_embed['cell_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]

# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(557, 768)
(557, 768)


In [95]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 39.4003, Val Loss = 0.6924
Epoch 2: Train Loss = 39.0547, Val Loss = 0.6925
Epoch 3: Train Loss = 38.6118, Val Loss = 0.6960
Epoch 4: Train Loss = 38.2321, Val Loss = 0.6921
Epoch 5: Train Loss = 37.9376, Val Loss = 0.6926
Epoch 6: Train Loss = 37.6668, Val Loss = 0.6922
Epoch 7: Train Loss = 37.4220, Val Loss = 0.6933
Epoch 8: Train Loss = 37.2035, Val Loss = 0.6951
Epoch 9: Train Loss = 36.9056, Val Loss = 0.6951
Epoch 10: Train Loss = 36.6404, Val Loss = 0.6969
Epoch 11: Train Loss = 36.5680, Val Loss = 0.6979
Epoch 12: Train Loss = 36.2982, Val Loss = 0.6992
Epoch 13: Train Loss = 36.0673, Val Loss = 0.7005
Epoch 14: Train Loss = 35.9033, Val Loss = 0.7017
Early stopping at epoch 14

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.4902    0.4310    0.4587        58
           1     0.4590    0.5185    0.4870        54

    accuracy                         0.4732       112
   mac

### 2.1.3 CellPLM

In [ ]:
method = 'CellPLM'
niche_key = 'niche_type'
emb_key = 'emb'

#### 2.1.3.1 Retrieve Embeddings

In [ ]:
# Cell embeddings
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

#### 2.1.3.2 Train Niche Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

##### 2.1.3.2.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.3.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate knn niche transfer classifier
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_knn_classifer_predicted_niches.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.3.3 Train Cell Type Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

##### 2.1.3.3.1 Linear Classifier

In [ ]:
# Retrieve features and labels
cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs['sample'] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs['sample'] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    cell_type_features_train = cell_type_features[train_indices]
    cell_type_features_val = cell_type_features[val_indices]
    cell_type_features_test = cell_type_features[test_indices]
    cell_type_labels_train = cell_type_labels[train_indices]
    cell_type_labels_val = cell_type_labels[val_indices]
    cell_type_labels_test = cell_type_labels[test_indices]

    # Train and evaluate linear cell type transfer classifier
    pred_cell_type_labels_test, _, _, model = linear_classifier(
        features_train=cell_type_features_train,
        labels_train=cell_type_labels_train,
        features_val=cell_type_features_val,
        labels_val=cell_type_labels_val,
        features_test=cell_type_features_test,
        labels_test=cell_type_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )

    #print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted cell type labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_linear_classifer_predicted_cell_types_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_linear',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.3.3.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate knn cell type transfer classifier
pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_knn_classifer_predicted_cell_types.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.3.4 Train Neighborhood Composition Classifiers

In [ ]:
# Compute spatial neighborhood graph
adata_batch_list = []
batches = adata_emb.obs[batch_key].unique().tolist()

for batch in batches:
    adata_batch = adata_emb[adata_emb.obs[batch_key] == batch]
    print('Computing spatial neighborhood...')
    # Construct neighbor graph
    adata_batch = construct_neighbor_graph(
        adata_batch,
        n_neighs=n_neighs,
        radius=None,
        delaunay=None,
        include_self_loop=False,
        compute_neighbor_counts=False)
    adata_batch_list.append(adata_batch)

adata_combined = sc.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
connectivities = sp.vstack(batch_connectivities)
adata_combined.obsp['spatial_connectivities'] = connectivities

composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata_emb.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get neighborhood composition labels and features
    composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
    cell_type_composition_labels = composition_df.values
    niche_features = adata_emb.obsm[emb_key]
    
    # Get train and validation indices
    train_val_indices = (adata_emb.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_composition_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        #stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_emb.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
    cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
    cell_type_composition_labels_test = cell_type_composition_labels[test_indices]
    
    pred_cell_type_composition_labels, model = linear_regressor(
        features_train=niche_features_train,
        labels_train=cell_type_composition_labels_train,
        features_val=niche_features_val,
        labels_val=cell_type_composition_labels_val,
        features_test=niche_features_test,
        labels_test=cell_type_composition_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_neighborhood_composition_linear_classifier_cls_report_seed_{seed}.txt'
    )
    
    print(pred_cell_type_composition_labels)

### 2.1.4 Nicheformer

In [ ]:
method = 'nicheformer'
niche_key = 'niche_type'
emb_key = 'X_niche_embeddings'

#### 2.1.4.1 Retrieve Embeddings

In [ ]:
# Cell embeddings
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

#### 2.1.4.2 Train Niche Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

##### 2.1.4.2.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.4.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate knn niche transfer classifier
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_knn_classifer_predicted_niches.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.4.3 Train Cell Type Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

##### 2.1.4.3.1 Linear Classifier

In [ ]:
# Retrieve features and labels
cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs['sample'] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs['sample'] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    cell_type_features_train = cell_type_features[train_indices]
    cell_type_features_val = cell_type_features[val_indices]
    cell_type_features_test = cell_type_features[test_indices]
    cell_type_labels_train = cell_type_labels[train_indices]
    cell_type_labels_val = cell_type_labels[val_indices]
    cell_type_labels_test = cell_type_labels[test_indices]

    # Train and evaluate linear cell type transfer classifier
    pred_cell_type_labels_test, _, _, model = linear_classifier(
        features_train=cell_type_features_train,
        labels_train=cell_type_labels_train,
        features_val=cell_type_features_val,
        labels_val=cell_type_labels_val,
        features_test=cell_type_features_test,
        labels_test=cell_type_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )

    #print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted cell type labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_linear_classifer_predicted_cell_types_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_linear',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.4.3.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate knn cell type transfer classifier
pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_knn_classifer_predicted_cell_types.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.4.4 Train Neighborhood Composition Classifiers

In [ ]:
# Compute spatial neighborhood graph
adata_batch_list = []
batches = adata_emb.obs[batch_key].unique().tolist()

for batch in batches:
    adata_batch = adata_emb[adata_emb.obs[batch_key] == batch]
    print('Computing spatial neighborhood...')
    # Construct neighbor graph
    adata_batch = construct_neighbor_graph(
        adata_batch,
        n_neighs=n_neighs,
        radius=None,
        delaunay=None,
        include_self_loop=False,
        compute_neighbor_counts=False)
    adata_batch_list.append(adata_batch)

adata_combined = sc.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
connectivities = sp.vstack(batch_connectivities)
adata_combined.obsp['spatial_connectivities'] = connectivities

composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata_emb.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get neighborhood composition labels and features
    composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
    cell_type_composition_labels = composition_df.values
    niche_features = adata_emb.obsm[emb_key]
    
    # Get train and validation indices
    train_val_indices = (adata_emb.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_composition_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        #stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_emb.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
    cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
    cell_type_composition_labels_test = cell_type_composition_labels[test_indices]
    
    pred_cell_type_composition_labels, model = linear_regressor(
        features_train=niche_features_train,
        labels_train=cell_type_composition_labels_train,
        features_val=niche_features_val,
        labels_val=cell_type_composition_labels_val,
        features_test=niche_features_test,
        labels_test=cell_type_composition_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_neighborhood_composition_linear_classifier_cls_report_seed_{seed}.txt'
    )
    
    print(pred_cell_type_composition_labels)

### 2.1.5 Novae

In [ ]:
method = 'Novae'
niche_key = 'niche_type'
emb_key = 'novae_latent'

#### 2.1.5.1 Retrieve Embeddings

In [ ]:
# Cell embeddings
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

#### 2.1.5.2 Train Niche Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

##### 2.1.5.2.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.5.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate knn niche transfer classifier
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_knn_classifer_predicted_niches.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.5.3 Train Cell Type Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

##### 2.1.5.3.1 Linear Classifier

In [ ]:
# Retrieve features and labels
cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs['sample'] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs['sample'] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    cell_type_features_train = cell_type_features[train_indices]
    cell_type_features_val = cell_type_features[val_indices]
    cell_type_features_test = cell_type_features[test_indices]
    cell_type_labels_train = cell_type_labels[train_indices]
    cell_type_labels_val = cell_type_labels[val_indices]
    cell_type_labels_test = cell_type_labels[test_indices]

    # Train and evaluate linear cell type transfer classifier
    pred_cell_type_labels_test, _, _, model = linear_classifier(
        features_train=cell_type_features_train,
        labels_train=cell_type_labels_train,
        features_val=cell_type_features_val,
        labels_val=cell_type_labels_val,
        features_test=cell_type_features_test,
        labels_test=cell_type_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )

    #print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted cell type labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_linear_classifer_predicted_cell_types_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_linear',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.5.3.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate knn cell type transfer classifier
pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_knn_classifer_predicted_cell_types.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.5.4 Train Neighborhood Composition Classifiers

In [ ]:
# Compute spatial neighborhood graph
adata_batch_list = []
batches = adata_emb.obs[batch_key].unique().tolist()

for batch in batches:
    adata_batch = adata_emb[adata_emb.obs[batch_key] == batch]
    print('Computing spatial neighborhood...')
    # Construct neighbor graph
    adata_batch = construct_neighbor_graph(
        adata_batch,
        n_neighs=n_neighs,
        radius=None,
        delaunay=None,
        include_self_loop=False,
        compute_neighbor_counts=False)
    adata_batch_list.append(adata_batch)

adata_combined = sc.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
connectivities = sp.vstack(batch_connectivities)
adata_combined.obsp['spatial_connectivities'] = connectivities

composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata_emb.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get neighborhood composition labels and features
    composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
    cell_type_composition_labels = composition_df.values
    niche_features = adata_emb.obsm[emb_key]
    
    # Get train and validation indices
    train_val_indices = (adata_emb.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_composition_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        #stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_emb.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
    cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
    cell_type_composition_labels_test = cell_type_composition_labels[test_indices]
    
    pred_cell_type_composition_labels, model = linear_regressor(
        features_train=niche_features_train,
        labels_train=cell_type_composition_labels_train,
        features_val=niche_features_val,
        labels_val=cell_type_composition_labels_val,
        features_test=niche_features_test,
        labels_test=cell_type_composition_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_neighborhood_composition_linear_classifier_cls_report_seed_{seed}.txt'
    )
    
    print(pred_cell_type_composition_labels)

### 2.1.6 scGPT-spatial

In [ ]:
method = 'scGPT-spatial'
niche_key = 'niche_type'
emb_key = 'X'

#### 2.1.6.1 Retrieve Embeddings

In [ ]:
# Cell embeddings
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

#### 2.1.6.2 Train Niche Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

##### 2.1.6.2.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.6.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate knn niche transfer classifier
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_knn_classifer_predicted_niches.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.6.3 Train Cell Type Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

##### 2.1.6.3.1 Linear Classifier

In [ ]:
# Retrieve features and labels
cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs['sample'] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs['sample'] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    cell_type_features_train = cell_type_features[train_indices]
    cell_type_features_val = cell_type_features[val_indices]
    cell_type_features_test = cell_type_features[test_indices]
    cell_type_labels_train = cell_type_labels[train_indices]
    cell_type_labels_val = cell_type_labels[val_indices]
    cell_type_labels_test = cell_type_labels[test_indices]

    # Train and evaluate linear cell type transfer classifier
    pred_cell_type_labels_test, _, _, model = linear_classifier(
        features_train=cell_type_features_train,
        labels_train=cell_type_labels_train,
        features_val=cell_type_features_val,
        labels_val=cell_type_labels_val,
        features_test=cell_type_features_test,
        labels_test=cell_type_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )

    #print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted cell type labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_linear_classifer_predicted_cell_types_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_linear',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.6.3.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate knn cell type transfer classifier
pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_knn_classifer_predicted_cell_types.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.6.4 Train Neighborhood Composition Classifiers

In [ ]:
# Compute spatial neighborhood graph
adata_batch_list = []
batches = adata_emb.obs[batch_key].unique().tolist()

for batch in batches:
    adata_batch = adata_emb[adata_emb.obs[batch_key] == batch]
    print('Computing spatial neighborhood...')
    # Construct neighbor graph
    adata_batch = construct_neighbor_graph(
        adata_batch,
        n_neighs=n_neighs,
        radius=None,
        delaunay=None,
        include_self_loop=False,
        compute_neighbor_counts=False)
    adata_batch_list.append(adata_batch)

adata_combined = sc.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
connectivities = sp.vstack(batch_connectivities)
adata_combined.obsp['spatial_connectivities'] = connectivities

composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata_emb.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get neighborhood composition labels and features
    composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
    cell_type_composition_labels = composition_df.values
    niche_features = adata_emb.obsm[emb_key]
    
    # Get train and validation indices
    train_val_indices = (adata_emb.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_composition_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        #stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_emb.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
    cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
    cell_type_composition_labels_test = cell_type_composition_labels[test_indices]
    
    pred_cell_type_composition_labels, model = linear_regressor(
        features_train=niche_features_train,
        labels_train=cell_type_composition_labels_train,
        features_val=niche_features_val,
        labels_val=cell_type_composition_labels_val,
        features_test=niche_features_test,
        labels_test=cell_type_composition_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_neighborhood_composition_linear_classifier_cls_report_seed_{seed}.txt'
    )
    
    print(pred_cell_type_composition_labels)

### 2.1.7 Random

In [27]:
method = 'random'

#### 2.1.7.1 Ligand-Receptor Prediction

##### 2.1.7.1.1 CellPhoneDB

In [36]:
lr_database = 'cellphonedb'

# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_list[0],
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_list[0],
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

Downloaded cellphonedb.zip into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded complex_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded gene_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded interaction_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded protein_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0
Downloaded uniprot_synonyms.tsv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0/sources
Downloaded transcription_factor_input.csv into /nfs/team361/sb75/nichejepa-reproducibility/artifacts/cellphonedb/v5.0.0/sources

               ligand         receptor
2     ENSG00000039068  ENSG00000139187
4     ENSG00000170558  ENSG00000104921
232   ENSG00000090339  ENSG00000005844
237   ENSG00000108622  ENSG00000090659
239   ENSG00000076662  E

In [38]:
# Retrieve Ensembl IDs for cell and neighborhood
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()
all_gene_ensembl_id = list(set(cell_gene_ensembl_id + neighborhood_gene_ensembl_id))

# Create random embeddings for each Ensembl ID
random_state = 0
np.random.seed(random_state)
emsemblid_embedding_mapping = {ensembl_id: np.random.randn(384) for ensembl_id in all_gene_ensembl_id}

# Retrieve random embeddings for ligand-receptor pairs
neighborhood_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in neighborhood_gene_ensembl_id])
cell_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in cell_gene_ensembl_id])

In [39]:
# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(207, 768)
(207, 768)


In [40]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 15.0937, Val Loss = 0.6984
Epoch 2: Train Loss = 14.8575, Val Loss = 0.6958
Epoch 3: Train Loss = 14.5861, Val Loss = 0.6920
Epoch 4: Train Loss = 14.3136, Val Loss = 0.6891
Epoch 5: Train Loss = 14.0069, Val Loss = 0.6864
Epoch 6: Train Loss = 13.7935, Val Loss = 0.6846
Epoch 7: Train Loss = 13.5161, Val Loss = 0.6817
Epoch 8: Train Loss = 13.3211, Val Loss = 0.6793
Epoch 9: Train Loss = 13.0875, Val Loss = 0.6776
Epoch 10: Train Loss = 12.9159, Val Loss = 0.6760
Epoch 11: Train Loss = 12.7129, Val Loss = 0.6746
Epoch 12: Train Loss = 12.4686, Val Loss = 0.6725
Epoch 13: Train Loss = 12.3460, Val Loss = 0.6715
Epoch 14: Train Loss = 12.1729, Val Loss = 0.6704
Epoch 15: Train Loss = 12.0136, Val Loss = 0.6691
Epoch 16: Train Loss = 11.8213, Val Loss = 0.6682
Epoch 17: Train Loss = 11.7176, Val Loss = 0.6672
Epoch 18: Train Loss = 11.5597, Val Loss = 0.6665
Epoch 19: Train Loss = 11.4081, Val Loss = 0.6663
Epoch 20: Train Loss = 11.3035, Val Loss

#### 2.1.7.2 OmniPath

In [41]:
lr_database = 'omnipath'

# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_list[0],
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_list[0],
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

                ligand         receptor
66     ENSG00000187266  ENSG00000138741
84     ENSG00000010810  ENSG00000133107
111    ENSG00000010810  ENSG00000137672
183    ENSG00000010810  ENSG00000111199
337    ENSG00000133116  ENSG00000165125
...                ...              ...
88693  ENSG00000113520  ENSG00000166888
89474  ENSG00000067182  ENSG00000065559
90565  ENSG00000019991  ENSG00000100030
90821  ENSG00000125538  ENSG00000105397
93513  ENSG00000136244  ENSG00000168610

[1159 rows x 2 columns]


In [42]:
# Retrieve Ensembl IDs for cell and neighborhood
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()
all_gene_ensembl_id = list(set(cell_gene_ensembl_id + neighborhood_gene_ensembl_id))

# Create random embeddings for each Ensembl ID
random_state = 0
np.random.seed(random_state)
emsemblid_embedding_mapping = {ensembl_id: np.random.randn(384) for ensembl_id in all_gene_ensembl_id}

# Retrieve random embeddings for ligand-receptor pairs
neighborhood_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in neighborhood_gene_ensembl_id])
cell_gene_emb_avg = np.array([emsemblid_embedding_mapping[ensembl_id] for ensembl_id in cell_gene_ensembl_id])

In [43]:
# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

(579, 768)
(579, 768)


In [44]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

Using device: cuda
Epoch 1: Train Loss = 41.4665, Val Loss = 0.7009
Epoch 2: Train Loss = 40.5080, Val Loss = 0.7045
Epoch 3: Train Loss = 39.6605, Val Loss = 0.7089
Epoch 4: Train Loss = 38.9127, Val Loss = 0.7105
Epoch 5: Train Loss = 38.2261, Val Loss = 0.7159
Epoch 6: Train Loss = 37.6265, Val Loss = 0.7188
Epoch 7: Train Loss = 37.0368, Val Loss = 0.7238
Epoch 8: Train Loss = 36.5304, Val Loss = 0.7276
Epoch 9: Train Loss = 36.0415, Val Loss = 0.7296
Epoch 10: Train Loss = 35.5925, Val Loss = 0.7356
Epoch 11: Train Loss = 35.1571, Val Loss = 0.7392
Early stopping at epoch 11

--- Evaluation Report on Test Set ---
              precision    recall  f1-score   support

           0     0.5517    0.5000    0.5246        64
           1     0.4483    0.5000    0.4727        52

    accuracy                         0.5000       116
   macro avg     0.5000    0.5000    0.4987       116
weighted avg     0.5054    0.5000    0.5013       116


--- Evaluation Report saved. ---
Using device:

## 2.2 chl59-8b_1p (NanoString CosMx Human Lung NSCLC)

In [96]:
# Dataset variables
dataset_name = 'chl59-8b_1p'
cell_type_key = 'cell_type'
niche_key = 'niche'
sample_key = 'batch'
file_paths = [
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung12+SMI+Flat+data.tar.h5ad',
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung13+SMI+Flat+data.tar.h5ad', # unannotated
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung5_Rep1+SMI+Flat+data.tar.h5ad',
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung5_Rep2+SMI+Flat+data.tar.h5ad',
    #f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung5_Rep3+SMI+Flat+data.tar.h5ad', # unannotated
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung6+SMI+Flat+data.tar.h5ad',
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung9_Rep1+SMI+Flat+data.tar.h5ad',
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/Lung9_Rep2+SMI+Flat+data.tar.h5ad']
spot_size = 100

### 2.2.1 Visualize Ground Truth Niches

In [97]:
adata_list = []

for file_path in file_paths:
    # Load AnnData
    adata = sc.read_h5ad(file_path)

    # Harmonize AnnData
    adata = harmonize_adata(adata)
    adata.obs[sample_key] = adata.uns[sample_key]
    adata_list.append(adata)

STEP 1: ADDING ENSEMBL IDS...
Adding ensembl IDs from release 110...


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Number of genes with matching ensembl IDs:           946.
Number of genes skipped:           0.
STEP 2: ADDING SPECIAL VALUES...
STEP 1: ADDING ENSEMBL IDS...
Adding ensembl IDs from release 110...


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Number of genes with matching ensembl IDs:           946.
Number of genes skipped:           0.
STEP 2: ADDING SPECIAL VALUES...
STEP 1: ADDING ENSEMBL IDS...
Adding ensembl IDs from release 110...


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


... (3 outputs left out)

INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Number of genes with matching ensembl IDs:           946.
Number of genes skipped:           0.
STEP 2: ADDING SPECIAL VALUES...
STEP 1: ADDING ENSEMBL IDS...
Adding ensembl IDs from release 110...


INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.cdna.all.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.ncrna.fa.gz.pickle
INFO:pyensembl.sequence_data:Loaded sequence dictionary from /nfs/users/nfs_s/sb75/.cache/pyensembl/GRCh38/ensembl110/Homo_sapiens.GRCh38.pep.all.fa.gz.pickle


Number of genes with matching ensembl IDs:           946.
Number of genes skipped:           0.
STEP 2: ADDING SPECIAL VALUES...


In [ ]:
# Get unique niche labels
set(adata_list[0].obs[niche_key].unique().tolist() +
    adata_list[1].obs[niche_key].unique().tolist() +
    adata_list[2].obs[niche_key].unique().tolist() +
    adata_list[3].obs[niche_key].unique().tolist() +
    adata_list[4].obs[niche_key].unique().tolist() +
    adata_list[5].obs[niche_key].unique().tolist())

In [ ]:
# Define niche palette
label2niche = {
    '0': '1- Tumor (stroma border)',
    '2': '2- Tumor interior',
    '5': '3- Tumor (neutrophil border)',
    '7': '4- Tumor interior',
    '11': '5- Infiltrated tumor',
    '3': '6- Tumor-infiltrating neutrophils',
    '4': '7- Neutrophil expansion',
    '1': '8- Stroma',
    '6': '9- Plasmablast rich stroma',
    '9': '10- Lymphoid rich stroma',
    '8': '11- Lymphoid aggregates',
    '10': '12- Macrophage rich stroma'
}

niche_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

In [ ]:
# Visualize ground truth niche labels in training samples
batch = list(set(adata_list[0].obs[sample_key].tolist()))[0]
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[0],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)

batch = list(set(adata_list[1].obs[sample_key].tolist()))[0]
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[1],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)

batch = list(set(adata_list[2].obs[sample_key].tolist()))[0]
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[2],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name) 

batch = list(set(adata_list[3].obs[sample_key].tolist()))[0]
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[3],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name) 

batch = list(set(adata_list[4].obs[sample_key].tolist()))[0]
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[4],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name) 

In [ ]:
# Visualize ground truth niche labels in test sample
batch = list(set(adata_list[5].obs[sample_key].tolist()))[0]
file_name = f'{dataset_name}_{batch}_niche_label_transfer_ground_truth_niches.svg'
sc.pl.spatial(
    adata_list[5],
    color=niche_key,
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name) 

### 2.2.2 NicheJEPA

In [ ]:
method = 'NicheJEPA'
niche_key = 'niche_type'
emb_key = 'neighborhood_emb'

#### 2.2.2.1 Retrieve Embeddings

In [98]:
# Cell and neighborhood embeddings
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded_{model_timestamp}.h5ad')

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [99]:
# Gene embeddings
dataset_list = []

for file_path in file_paths:
    # Load tokenized AnnData or tokenize data
    dataset_path = f'{artifacts_folder_path}/{dataset_name}/{file_path.split("/")[-1].split(".")[0]}.dataset'
    if os.path.exists(dataset_path) and not force_retokenize:
        dataset = load_from_disk(dataset_path)
    else:
        # Tokenize AnnData
        dataset = tokenize_adata(
            adata,
            model_folder_path,
            nproc=nproc,
            processing_mode='parallel')

        # Save tokenized data
        os.makedirs(f'{artifacts_folder_path}/{dataset_name}', exist_ok=True)
        dataset.save_to_disk(
            dataset_path,
            num_shards=num_shards)

    dataset_list.append(dataset)

full_dataset = concatenate_datasets(dataset_list)

#### 2.2.2.2 Train Niche Transfer Classifiers

In [100]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

niche
1- Tumor (stroma border)             86677
8- Stroma                            80404
2- Tumor interior                    60056
6- Tumor-infiltrating neutrophils    43489
7- Neutrophil expansion              43190
3- Tumor (neutrophil border)         42843
9- Plasmablast rich stroma           41651
4- Tumor interior                    40059
11- Lymphoid aggregates              34197
10- Lymphoid rich stroma             26911
12- Macrophage rich stroma           20425
5- Infiltrated tumor                 14059
Name: count, dtype: int64


##### 2.1.2.2.1 Linear Classifier

In [101]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [104]:
adata_list[5].obs[sample_key]

1_5        batch5
1_7        batch5
1_8        batch5
1_9        batch5
1_10       batch5
            ...  
45_3035    batch5
45_3036    batch5
45_3037    batch5
45_3038    batch5
45_3039    batch5
Name: batch, Length: 144980, dtype: object

In [105]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'batch5').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'batch5').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

Using device: cuda
Epoch 1: Train Loss = 2356.3826, Val Loss = 0.5309
Epoch 2: Train Loss = 1906.2413, Val Loss = 0.4991
Epoch 3: Train Loss = 1824.3619, Val Loss = 0.4846
Epoch 4: Train Loss = 1779.5842, Val Loss = 0.4758
Epoch 5: Train Loss = 1749.6842, Val Loss = 0.4696
Epoch 6: Train Loss = 1727.5282, Val Loss = 0.4647
Epoch 7: Train Loss = 1709.8149, Val Loss = 0.4609
Epoch 8: Train Loss = 1695.5035, Val Loss = 0.4578
Epoch 9: Train Loss = 1683.2511, Val Loss = 0.4551
Epoch 10: Train Loss = 1672.8596, Val Loss = 0.4528
Epoch 11: Train Loss = 1663.6088, Val Loss = 0.4511
Epoch 12: Train Loss = 1655.5263, Val Loss = 0.4489
Epoch 13: Train Loss = 1648.0899, Val Loss = 0.4474
Epoch 14: Train Loss = 1641.5659, Val Loss = 0.4456
Epoch 15: Train Loss = 1635.4216, Val Loss = 0.4441
Epoch 16: Train Loss = 1629.7366, Val Loss = 0.4429
Epoch 17: Train Loss = 1624.4153, Val Loss = 0.4416
Epoch 18: Train Loss = 1619.5016, Val Loss = 0.4405
Epoch 19: Train Loss = 1614.7886, Val Loss = 0.4394
Ep

ValueError: max() arg is an empty sequence

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_{model_timestamp}_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.2.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'batch5').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'batch5').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate knn niche transfer classifier
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_{model_timestamp}_niche_label_transfer_knn_classifer_predicted_niches.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.2.2.3 Train Cell Type Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

##### 2.1.2.3.1 Linear Classifier

In [ ]:
# Retrieve features and labels
cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm['cell_emb']

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs['sample'] != 'batch5').to_numpy().nonzero()[0]
    train_val_labels = cell_type_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs['sample'] == 'batch5').to_numpy().nonzero()[0]
    
    # Split features and labels
    cell_type_features_train = cell_type_features[train_indices]
    cell_type_features_val = cell_type_features[val_indices]
    cell_type_features_test = cell_type_features[test_indices]
    cell_type_labels_train = cell_type_labels[train_indices]
    cell_type_labels_val = cell_type_labels[val_indices]
    cell_type_labels_test = cell_type_labels[test_indices]

    # Train and evaluate linear cell type transfer classifier
    pred_cell_type_labels_test, _, _, model = linear_classifier(
        features_train=cell_type_features_train,
        labels_train=cell_type_labels_train,
        features_val=cell_type_features_val,
        labels_val=cell_type_labels_val,
        features_test=cell_type_features_test,
        labels_test=cell_type_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_cell_type_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )

    #print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted cell type labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_{model_timestamp}_cell_type_label_transfer_linear_classifer_predicted_cell_types_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_linear',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.2.3.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'batch5').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'batch5').to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate knn cell type transfer classifier
pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_cell_type_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_{model_timestamp}_cell_type_label_transfer_knn_classifer_predicted_cell_types.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.2.2.4 Train Neighborhood Composition Classifiers

In [ ]:
# Compute spatial neighborhood graph
adata_batch_list = []
batches = adata_emb.obs[batch_key].unique().tolist()

for batch in batches:
    adata_batch = adata_emb[adata_emb.obs[batch_key] == batch]
    print('Computing spatial neighborhood...')
    # Construct neighbor graph
    adata_batch = construct_neighbor_graph(
        adata_batch,
        n_neighs=n_neighs,
        radius=None,
        delaunay=None,
        include_self_loop=False,
        compute_neighbor_counts=False)
    adata_batch_list.append(adata_batch)

adata_combined = sc.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
connectivities = sp.vstack(batch_connectivities)
adata_combined.obsp['spatial_connectivities'] = connectivities

composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata_emb.obsm['neighborhood_emb']

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get neighborhood composition labels and features
    composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
    cell_type_composition_labels = composition_df.values
    niche_features = adata_emb.obsm['neighborhood_emb']
    
    # Get train and validation indices
    train_val_indices = (adata_emb.obs[sample_key] != 'batch5').to_numpy().nonzero()[0]
    train_val_labels = cell_type_composition_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        #stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_emb.obs[sample_key] == 'batch5').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
    cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
    cell_type_composition_labels_test = cell_type_composition_labels[test_indices]
    
    pred_cell_type_composition_labels, model = linear_regressor(
        features_train=niche_features_train,
        labels_train=cell_type_composition_labels_train,
        features_val=niche_features_val,
        labels_val=cell_type_composition_labels_val,
        features_test=niche_features_test,
        labels_test=cell_type_composition_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_neighborhood_composition_linear_classifier_cls_report_seed_{seed}.txt'
    )
    
    print(pred_cell_type_composition_labels)

#### 2.2.2.5 Ligand-Receptor Prediction

##### 2.1.2.5.1 CellPhoneDB

In [ ]:
lr_database = 'cellphonedb'

In [ ]:
# Get Ensembl IDs
adata_emb.var = adata_list[0].var

In [ ]:
# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_emb,
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_emb,
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

In [ ]:
# Retrieve cell gene embeddings for receptors and neighborhood gene embeddings for ligands
file_path = f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_average_gene_embed_{lr_database}.pkl'

if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        output_average_gene_embed = pickle.load(f)
else:
    cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
    neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()
    
    output_average_gene_embed = get_average_gene_embed(
        dataset=full_dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        cell_gene_ensembl_id=cell_gene_ensembl_id,
        neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
    )
    
    with open(file_path, "wb") as f:
        pickle.dump(output_average_gene_embed, f)

In [ ]:
# Filter embeddings with too few occurrences in either neighborhood or cell gene
min_count_threshhold = 10

neighborhood_gene_emb_avg = output_average_gene_embed['neighborhood_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]
cell_gene_emb_avg = output_average_gene_embed['cell_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]

# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

##### 2.1.2.5.2 OmniPath

In [ ]:
lr_database = 'omnipath'

In [ ]:
# Get Ensembl IDs
adata_emb.var = adata_list[0].var

In [ ]:
# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata_emb,
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata_emb,
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')
print(lr_interaction_df)

# Randomly shuffle rows
lr_interaction_df = lr_interaction_df.sample(
    frac=1, random_state=0).reset_index(drop=True)

In [ ]:
# Retrieve cell gene embeddings for receptors and neighborhood gene embeddings for ligands
file_path = f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_average_gene_embed_{lr_database}.pkl'

if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        output_average_gene_embed = pickle.load(f)
else:
    cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
    neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()
    
    output_average_gene_embed = get_average_gene_embed(
        dataset=full_dataset,
        model_folder_path=model_folder_path,
        emb_layer=emb_layer,
        batch_size=batch_size,
        pin_memory=pin_memory,
        num_workers=num_workers,
        cell_gene_ensembl_id=cell_gene_ensembl_id,
        neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
    )
    
    with open(file_path, "wb") as f:
        pickle.dump(output_average_gene_embed, f)

In [ ]:
# Filter embeddings with too few occurrences in either neighborhood or cell gene
min_count_threshhold = 10

neighborhood_gene_emb_avg = output_average_gene_embed['neighborhood_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]
cell_gene_emb_avg = output_average_gene_embed['cell_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]

# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

### Use second half of ligand-receptor pairs as negative pairs ###
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    train_ratio = 0.8
    test_ratio = 0.1
    val_ratio = 0.1
    
    # --- Splitting using sklearn ---
    # First, split off the test set
    ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
        ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)
    
    # Then split the remaining temp set into train and val
    # The new val ratio is relative to the remaining data
    val_relative_ratio = val_ratio / (train_ratio + val_ratio)
    
    ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
        ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)
    
    # Train and evaluate linear ligand-receptor classifier
    pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
        features_train=ligand_receptor_features_train,
        labels_train=ligand_receptor_labels_train,
        features_val=ligand_receptor_features_val,
        labels_val=ligand_receptor_labels_val,
        features_test=ligand_receptor_features_test,
        labels_test=ligand_receptor_labels_test,
        n_epochs=500,
        batch_size=16,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_cls_report_seed_{seed}.txt'
        )

    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_pred_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(pred_lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_labels_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_labels, f)
    with open(f'{benchmarking_artifacts_folder_path}/results/{method}_{model_timestamp}_{dataset_name}_ligand_receptor_prediction_linear_classifier_lr_logits_seed_{seed}.txt', "wb") as f:
        pickle.dump(lr_logits, f)

In [ ]:
# Harmonize and tokenize data and retrieve embeddings
force_retokenize = False
force_recompute_output_embed = False

adata_list = []
dataset_list = []
output_embed_list = []

for file_path in file_paths:
    # Load AnnData
    adata = sc.read_h5ad(file_path)

    # Harmonize AnnData
    adata = harmonize_adata(adata)
    adata_list.append(adata)

    # Load tokenized AnnData or tokenize data
    dataset_path = f'{artifacts_folder_path}/{dataset_name}/{file_path.split("/")[-1].split(".")[0]}.dataset'
    if os.path.exists(dataset_path) and not force_retokenize:
        dataset = load_from_disk(dataset_path)
        print('Filtering cells...')
        # Filter cells based on adata.obs['filter_pass']
        adata = filter_cells(adata)   
        
        print('Computing spatial neighborhood...')
        # Construct neighbor graph
        adata = construct_neighbor_graph(
            adata,
            n_neighs=n_neighs,
            radius=None,
            delaunay=None,
            include_self_loop=False,
            compute_neighbor_counts=False)
    else:
        # Tokenize AnnData
        dataset = tokenize_adata(
            adata,
            model_folder_path,
            nproc=nproc,
            processing_mode='parallel')

        # Save tokenized data
        os.makedirs(f'{artifacts_folder_path}/{dataset_name}', exist_ok=True)
        dataset.save_to_disk(
            dataset_path,
            num_shards=num_shards)
    dataset_list.append(dataset)

    # Retrieve cell and neighborhood embeddings
    output_embed_path = f'{artifacts_folder_path}/{dataset_name}/{model_timestamp}/{file_path.split("/")[-1].split(".")[0]}_output_embed.pkl'
    if os.path.exists(output_embed_path) and not force_recompute_output_embed:
        with open(output_embed_path, 'rb') as f:
            output_embed = pickle.load(f)
    else:
        # Compute embeddings
        output_embed = embed_dataset(
            dataset=dataset,
            model_folder_path=model_folder_path,
            emb_layer=emb_layer,
            agg_excluded_tokens=agg_excluded_tokens,
            top_k=top_k,
            batch_size=batch_size,
            pin_memory=pin_memory,
            num_workers=num_workers)

        # Save embeddings
        os.makedirs(f'{artifacts_folder_path}/{dataset_name}/{model_timestamp}', exist_ok=True)
        with open(output_embed_path, 'wb') as f:
            pickle.dump(output_embed, f)
    output_embed_list.append(output_embed)

### 2.2.3 CellPLM

### 2.2.4 Nicheformer

### 2.2.5 Novae

### 2.2.6 scGPT-spatial

In [ ]:
method = 'scGPT-spatial'
niche_key = 'niche'
emb_key = 'X'

#### 2.1.6.1 Retrieve Embeddings

In [ ]:
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

#### 2.1.6.2 Train Niche Transfer Classifiers

##### 2.1.6.2.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'batch5').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'batch5').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
file_name = f'{method}_{dataset_name}_batch19_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.6.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs['sample'] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs['sample'] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate linear niche transfer classifier
# 0.828 accuracy
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
file_name = f'{method}_{dataset_name}_batch19_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

In [ ]:
for i in range(len(adata_list)):
    adata_list[i].obs['batch'] = adata_list[i].uns['batch']

# Concatenate AnnData objects of all samples/batches
adata = sc.concat(adata_list, axis=0)

# Retrieve cell and neighborhood embeddings for all samples/batches
cell_emb = np.concatenate([output_embed['cell_emb'] for output_embed in output_embed_list], axis=0)
spatial_cell_emb = np.concatenate([output_embed['spatial_cell_emb'] for output_embed in output_embed_list], axis=0)
neighborhood_emb = np.concatenate([output_embed['neighborhood_emb'] for output_embed in output_embed_list], axis=0)

# Add embeddings to AnnData object
adata.obsm['cell_emb'] = cell_emb
adata.obsm['spatial_cell_emb'] = spatial_cell_emb
adata.obsm['neighborhood_emb'] = neighborhood_emb

# Add combined neighborhood graph
blocks = [a.obsp['spatial_connectivities'] for a in adata_list]
combined_graph = sp.block_diag(blocks)
adata.obsp['spatial_connectivities'] = combined_graph.tocsr()

# Write to disk
adata.write(f'{artifacts_folder_path}/{dataset_name}/{model_timestamp}/{file_path.split("/")[-1].split(".")[0]}.h5ad')

#### 2.2.2 Train Niche Transfer Classifier

In [ ]:
# Get unique niche labels
set(adata_list[0].obs[niche_key].unique().tolist() +
    adata_list[1].obs[niche_key].unique().tolist() +
    adata_list[2].obs[niche_key].unique().tolist() +
    adata_list[3].obs[niche_key].unique().tolist() +
    adata_list[4].obs[niche_key].unique().tolist() +
    adata_list[5].obs[niche_key].unique().tolist())

In [ ]:
adata_test = adata_list[-1]

In [ ]:
# Visualize ground truth niche labels in test sample
sc.pl.spatial(
    adata_test,
    color=niche_key,
    palette=custom_palette,
    spot_size=spot_size)   

In [ ]:
# Filter cells with missing labels
adata_filtered = adata[~adata.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

##### 2.2.1.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm['neighborhood_emb']

In [ ]:
# Get train and validation indices
train_val_indices = (adata_filtered.obs['batch'] != 'batch5').to_numpy().nonzero()[0]
train_val_labels = niche_labels[train_val_indices]
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=0.1,
    random_state=seed,
    stratify=train_val_labels
)

# Get test indices
test_indices = (adata_filtered.obs['batch'] == 'batch5').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_val = niche_features[val_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_val = niche_labels[val_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate linear niche transfer classifier
# 0.832 accuracy
pred_niche_labels_test, _, _, model = linear_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_val=niche_features_val,
    labels_val=niche_labels_val,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    n_epochs=500,
    batch_size=128,
    )

print(niche_categories[pred_niche_labels_test])

In [ ]:
adata_test = adata_filtered[adata_filtered.obs['batch'] == 'batch5']

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=custom_palette,
    spot_size=spot_size)   

##### 2.2.1.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs['batch'] != 'batch5').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs['batch'] == 'batch5').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate linear niche transfer classifier
# 0.789 accuracy
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test)

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=custom_palette,
    spot_size=spot_size)   

#### 2.2.3 Ligand-Receptor Prediction

In [ ]:
lr_database = 'cellphonedb'

In [ ]:
adata.var = adata_list[0].var

In [ ]:
# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata,
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata,
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')

print(lr_interaction_df)

In [ ]:
# Retrieve cell gene embeddings for receptors and neighborhood gene embeddings for ligands
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()

output_average_gene_embed = get_average_gene_embed(dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
# Filter embeddings with too few occurrences in either neighborhood or cell gene
min_count_threshhold = 10

neighborhood_gene_emb_avg = output_average_gene_embed['neighborhood_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]
cell_gene_emb_avg = output_average_gene_embed['cell_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]

# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

# Use second half og ligand-receptor pairs as negative pairs
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])


In [ ]:
seed = 0
train_ratio = 0.8
test_ratio = 0.1
val_ratio = 0.1

In [ ]:
# --- Splitting using sklearn ---
# First, split off the test set
ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
    ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)

# Then split the remaining temp set into train and val
# The new val ratio is relative to the remaining data
val_relative_ratio = val_ratio / (train_ratio + val_ratio)

ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
    ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)

In [ ]:
pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
    features_train=ligand_receptor_features_train,
    labels_train=ligand_receptor_labels_train,
    features_val=ligand_receptor_features_val,
    labels_val=ligand_receptor_labels_val,
    features_test=ligand_receptor_features_test,
    labels_test=ligand_receptor_labels_test,
    n_epochs=400,
    batch_size=16,
    )

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
from scipy.special import softmax

# Convert to numpy arrays
logits = np.stack(lr_logits)             # shape (N, 2)
labels = np.array(lr_labels)             # shape (N,)

# Apply softmax to get probabilities
probs = softmax(logits, axis=1)       # shape (N, 2)
pos_probs = probs[:, 1]               # probability of class 1

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(labels, pos_probs)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()

### 2.2.7 scGPT

## 2.3 xhk1007-41b_1p (Xenium Human Kidney RCC)

In [ ]:
# Dataset variables
dataset_name = 'xhk1007-CV1-41b_1p'
cell_type_key = 'level_1_cell_type'
sample_key = 'batch'
test_sample = 'batch14'
file_paths = [
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/CV1-KID-0-FT-3.h5ad',
    f'/nfs/team361/sb75/DATASETS/silver/{dataset_name}/CV1-KID-0-FT-1.h5ad']
spot_size = 30

In [ ]:
# Model variables
seed = 0
emb_layer = None
top_k = None
batch_size = 128
pin_memory = False
num_workers = 12
agg_excluded_tokens = None

train_ratio = 0.8
test_ratio = 0.1
val_ratio = 0.1

In [ ]:
# Tokenizer variables
num_shards = 32
nproc = 4
n_neighs = 10

#### 2.3.1 Retrieve Embeddings

In [ ]:
# Harmonize and tokenize data and retrieve embeddings
force_retokenize = False
force_recompute_output_embed = False

adata_list = []
dataset_list = []
output_embed_list = []

for file_path in file_paths:
    # Load AnnData
    adata = sc.read_h5ad(file_path)

    # Harmonize AnnData
    adata = harmonize_adata(adata)
    adata_list.append(adata)

    # Load tokenized AnnData or tokenize data
    dataset_path = f'{artifacts_folder_path}/{dataset_name}/{file_path.split("/")[-1].split(".")[0]}.dataset'
    if os.path.exists(dataset_path) and not force_retokenize:
        dataset = load_from_disk(dataset_path)
        print('Filtering cells...')
        # Filter cells based on adata.obs['filter_pass']
        adata = filter_cells(adata)   
        
        print('Computing spatial neighborhood...')
        # Construct neighbor graph
        adata = construct_neighbor_graph(
            adata,
            n_neighs=n_neighs,
            radius=None,
            delaunay=None,
            include_self_loop=False,
            compute_neighbor_counts=False)
    else:
        # Tokenize AnnData
        dataset = tokenize_adata(
            adata,
            model_folder_path,
            nproc=nproc,
            processing_mode='parallel')

        # Save tokenized data
        os.makedirs(f'{artifacts_folder_path}/{dataset_name}', exist_ok=True)
        dataset.save_to_disk(
            dataset_path,
            num_shards=num_shards)
    dataset_list.append(dataset)

    # Retrieve cell and neighborhood embeddings
    output_embed_path = f'{artifacts_folder_path}/{dataset_name}/{model_timestamp}/{file_path.split("/")[-1].split(".")[0]}_output_embed.pkl'
    if os.path.exists(output_embed_path) and not force_recompute_output_embed:
        with open(output_embed_path, 'rb') as f:
            output_embed = pickle.load(f)
    else:
        # Compute embeddings
        output_embed = embed_dataset(
            dataset=dataset,
            model_folder_path=model_folder_path,
            emb_layer=emb_layer,
            agg_excluded_tokens=agg_excluded_tokens,
            top_k=top_k,
            batch_size=batch_size,
            pin_memory=pin_memory,
            num_workers=num_workers)

        # Save embeddings
        os.makedirs(f'{artifacts_folder_path}/{dataset_name}/{model_timestamp}', exist_ok=True)
        with open(output_embed_path, 'wb') as f:
            pickle.dump(output_embed, f)
    output_embed_list.append(output_embed)

In [ ]:
for i in range(len(adata_list)):
    adata_list[i].obs['batch'] = adata_list[i].uns['batch']

# Concatenate AnnData objects of all samples/batches
adata = sc.concat(adata_list, axis=0)

# Retrieve cell and neighborhood embeddings for all samples/batches
cell_emb = np.concatenate([output_embed['cell_emb'] for output_embed in output_embed_list], axis=0)
spatial_cell_emb = np.concatenate([output_embed['spatial_cell_emb'] for output_embed in output_embed_list], axis=0)
neighborhood_emb = np.concatenate([output_embed['neighborhood_emb'] for output_embed in output_embed_list], axis=0)

# Add embeddings to AnnData object
adata.obsm['cell_emb'] = cell_emb
adata.obsm['spatial_cell_emb'] = spatial_cell_emb
adata.obsm['neighborhood_emb'] = neighborhood_emb

# Add combined neighborhood graph
blocks = [a.obsp['spatial_connectivities'] for a in adata_list]
combined_graph = sp.block_diag(blocks)
adata.obsp['spatial_connectivities'] = combined_graph.tocsr()

# Write to disk
adata.write(f'{artifacts_folder_path}/{dataset_name}/{model_timestamp}/{file_path.split("/")[-1].split(".")[0]}.h5ad')

#### 2.3.2 Train Cell Type Transfer Classifier

In [ ]:
# Get unique niche labels
set(adata_list[0].obs[cell_type_key].unique().tolist() +
    adata_list[1].obs[cell_type_key].unique().tolist())

In [ ]:
custom_palette = {
    'Endothelium': '#1f78b4',
    'Epithelium': '#33a02c',
    'Lymphoid': '#fb9a99',
    'Myeloid': '#e31a1c',
    'Stroma': '#a6cee3'
}

In [ ]:
# Visualize ground truth cell type labels in training samples
sc.pl.spatial(
    adata_list[0],
    color=cell_type_key,
    palette=custom_palette,
    spot_size=spot_size)   

In [ ]:
# Visualize ground truth cell type labels in test sample
sc.pl.spatial(
    adata_list[1],
    color=cell_type_key,
    palette=custom_palette,
    spot_size=spot_size)   

In [ ]:
# Filter cells with missing labels
adata_filtered = adata[~adata.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm['cell_emb']

# Get train and validation indices
train_val_indices = (adata_filtered.obs[sample_key] != test_sample).to_numpy().nonzero()[0]
train_val_labels = cell_type_labels[train_val_indices]
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=0.1,
    random_state=seed,
    stratify=train_val_labels
)

# Get test indices
test_indices = (adata_filtered.obs[sample_key] == test_sample).to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_val = cell_type_features[val_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_val = cell_type_labels[val_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate linear niche transfer classifier
# 0.891 accuracy
pred_cell_type_labels_test, _, _, model = linear_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_val=cell_type_features_val,
    labels_val=cell_type_labels_val,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    n_epochs=500,
    batch_size=128,
    )

print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
adata_test = adata_list[1]

In [ ]:
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted labels
sc.pl.spatial(
    adata_list[1],
    color=f'{cell_type_key}_pred_linear',
    palette=custom_palette,
    spot_size=spot_size)   

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != test_sample).to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == test_sample).to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test)

adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

# Visualize predicted niche labels in test sample
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=custom_palette,
    spot_size=30)   

#### 2.3.3 Train Neighborhood Composition Transfer Classifier

In [ ]:
# Get neighborhood composition labels and features
composition_df = compute_neighborhood_composition(adata, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata.obsm['neighborhood_emb']

# Get train and validation indices
train_val_indices = (adata.obs[sample_key] != test_sample).to_numpy().nonzero()[0]
train_val_labels = cell_type_composition_labels[train_val_indices]
train_indices, val_indices = train_test_split(
    train_val_indices,
    test_size=0.1,
    random_state=seed,
    #stratify=train_val_labels
)

# Get test indices
test_indices = (adata.obs[sample_key] == test_sample).to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_val = niche_features[val_indices]
niche_features_test = niche_features[test_indices]
cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
cell_type_composition_labels_test = cell_type_composition_labels[test_indices]

pred_cell_type_composition_labels, model = linear_regressor(
    features_train=niche_features_train,
    labels_train=cell_type_composition_labels_train,
    features_val=niche_features_val,
    labels_val=cell_type_composition_labels_val,
    features_test=niche_features_test,
    labels_test=cell_type_composition_labels_test,
    n_epochs=500,
    batch_size=128,
    )

print(pred_cell_type_composition_labels)

#### 2.3.4 Ligand-Receptor Prediction

In [ ]:
lr_database = 'cellphonedb'

In [ ]:
adata.var = adata_list[0].var

In [ ]:
# Retrieve ligand-receptor interaction pairs
if lr_database == 'omnipath': # 1159 pairs
    # Get OmniPath ligand-receptor interactions
    lr_interaction_df = get_adata_omnipath_lr_pairs(
        adata=adata,
        min_curation_effort=2)
elif lr_database == 'cellphonedb': # 414 pairs
    # Get CellPhoneDB ligand-receptor interactions
    lr_interaction_df = get_adata_cellphonedb_lr_pairs(
        adata=adata,
        save_folder_path=f'{artifacts_folder_path}/cellphonedb',
        cpdb_version='v5.0.0')

In [ ]:
lr_interaction_df

In [ ]:
# Retrieve cell gene embeddings for receptors and neighborhood gene embeddings for ligands
cell_gene_ensembl_id = lr_interaction_df['receptor'].values.tolist()
neighborhood_gene_ensembl_id = lr_interaction_df['ligand'].values.tolist()

output_average_gene_embed = get_average_gene_embed(dataset=dataset,
    model_folder_path=model_folder_path,
    emb_layer=emb_layer,
    batch_size=batch_size,
    pin_memory=pin_memory,
    num_workers=num_workers,
    cell_gene_ensembl_id=cell_gene_ensembl_id,
    neighborhood_gene_ensembl_id=neighborhood_gene_ensembl_id
)

In [ ]:
# Filter embeddings with too few occurrences in either neighborhood or cell gene
min_count_threshhold = 10

neighborhood_gene_emb_avg = output_average_gene_embed['neighborhood_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]
cell_gene_emb_avg = output_average_gene_embed['cell_gene_emb_average_per_data'][
    ((output_average_gene_embed['neighborhood_gene_emb_counts_per_data'] > min_count_threshhold) & (output_average_gene_embed['cell_gene_emb_counts_per_data'] > min_count_threshhold))[:, 0]]

# Use first half of ligand-receptor pairs as positive pairs
ligand_receptor_real_features = np.concatenate(
    [neighborhood_gene_emb_avg[:int(np.round(neighborhood_gene_emb_avg.shape[0]/2)),:],
     cell_gene_emb_avg[:int(np.round(cell_gene_emb_avg.shape[0]/2)),:]],
    axis=1)

# Use second half og ligand-receptor pairs as negative pairs
# Step 1: Generate a derangement (a permutation with no fixed points)
def generate_derangement(n):
    while True:
        perm = np.random.permutation(n)
        if not np.any(perm == np.arange(n)):
            return perm

# Get deranged indices for cell gene embeddings
deranged_indices = generate_derangement(cell_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:].shape[0])
cell_gene_emb_avg_shuffled = cell_gene_emb_avg[int(np.round(cell_gene_emb_avg.shape[0]/2)):,:][deranged_indices]

# Step 2: Concatenate deranged cell gene embeddings with neighborhood gene embeddings
ligand_receptor_fake_features = np.concatenate(
    [neighborhood_gene_emb_avg[int(np.round(neighborhood_gene_emb_avg.shape[0]/2)):,:], cell_gene_emb_avg_shuffled], axis=1)


if ligand_receptor_real_features.shape[0] < ligand_receptor_fake_features.shape[0]:
    ligand_receptor_fake_features = ligand_receptor_fake_features[:-1,:]
elif ligand_receptor_real_features.shape[0] > ligand_receptor_fake_features.shape[0]:
    ligand_receptor_real_features = ligand_receptor_real_features[:-1,:]
print(ligand_receptor_real_features.shape)
print(ligand_receptor_fake_features.shape)

ligand_receptor_features = np.concatenate([ligand_receptor_real_features, ligand_receptor_fake_features], axis=0)

# Define labels (1 for ground truth ligand-receptor pair and 0 for random sample)
ligand_receptor_labels = np.array([1] * ligand_receptor_real_features.shape[0] + [0] * ligand_receptor_fake_features.shape[0])


In [ ]:
# --- Splitting using sklearn ---
# First, split off the test set
ligand_receptor_features_temp, ligand_receptor_features_test, ligand_receptor_labels_temp, ligand_receptor_labels_test = train_test_split(
    ligand_receptor_features, ligand_receptor_labels, test_size=test_ratio, random_state=seed)

# Then split the remaining temp set into train and val
# The new val ratio is relative to the remaining data
val_relative_ratio = val_ratio / (train_ratio + val_ratio)

ligand_receptor_features_train, ligand_receptor_features_val, ligand_receptor_labels_train, ligand_receptor_labels_val = train_test_split(
    ligand_receptor_features_temp, ligand_receptor_labels_temp, test_size=val_relative_ratio, random_state=seed)

In [ ]:
pred_lr_labels, lr_labels, lr_logits, lr_model = linear_classifier(
    features_train=ligand_receptor_features_train,
    labels_train=ligand_receptor_labels_train,
    features_val=ligand_receptor_features_val,
    labels_val=ligand_receptor_labels_val,
    features_test=ligand_receptor_features_test,
    labels_test=ligand_receptor_labels_test,
    n_epochs=400,
    batch_size=16,
    )

In [ ]:
plot_roc_curve(logits=lr_logits, labels=lr_labels)

#### 2.1.6.1 Retrieve Embeddings

In [ ]:
# Cell embeddings
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

#### 2.1.6.2 Train Niche Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

##### 2.1.6.2.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.6.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate knn niche transfer classifier
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_knn_classifer_predicted_niches.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.6.3 Train Cell Type Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

##### 2.1.6.3.1 Linear Classifier

In [ ]:
# Retrieve features and labels
cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs['sample'] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs['sample'] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    cell_type_features_train = cell_type_features[train_indices]
    cell_type_features_val = cell_type_features[val_indices]
    cell_type_features_test = cell_type_features[test_indices]
    cell_type_labels_train = cell_type_labels[train_indices]
    cell_type_labels_val = cell_type_labels[val_indices]
    cell_type_labels_test = cell_type_labels[test_indices]

    # Train and evaluate linear cell type transfer classifier
    pred_cell_type_labels_test, _, _, model = linear_classifier(
        features_train=cell_type_features_train,
        labels_train=cell_type_labels_train,
        features_val=cell_type_features_val,
        labels_val=cell_type_labels_val,
        features_test=cell_type_features_test,
        labels_test=cell_type_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )

    #print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted cell type labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_linear_classifer_predicted_cell_types_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_linear',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.6.3.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate knn cell type transfer classifier
pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_knn_classifer_predicted_cell_types.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.6.4 Train Neighborhood Composition Classifiers

In [ ]:
# Compute spatial neighborhood graph
adata_batch_list = []
batches = adata_emb.obs[batch_key].unique().tolist()

for batch in batches:
    adata_batch = adata_emb[adata_emb.obs[batch_key] == batch]
    print('Computing spatial neighborhood...')
    # Construct neighbor graph
    adata_batch = construct_neighbor_graph(
        adata_batch,
        n_neighs=n_neighs,
        radius=None,
        delaunay=None,
        include_self_loop=False,
        compute_neighbor_counts=False)
    adata_batch_list.append(adata_batch)

adata_combined = sc.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
connectivities = sp.vstack(batch_connectivities)
adata_combined.obsp['spatial_connectivities'] = connectivities

composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata_emb.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get neighborhood composition labels and features
    composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
    cell_type_composition_labels = composition_df.values
    niche_features = adata_emb.obsm[emb_key]
    
    # Get train and validation indices
    train_val_indices = (adata_emb.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_composition_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        #stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_emb.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
    cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
    cell_type_composition_labels_test = cell_type_composition_labels[test_indices]
    
    pred_cell_type_composition_labels, model = linear_regressor(
        features_train=niche_features_train,
        labels_train=cell_type_composition_labels_train,
        features_val=niche_features_val,
        labels_val=cell_type_composition_labels_val,
        features_test=niche_features_test,
        labels_test=cell_type_composition_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_neighborhood_composition_linear_classifier_cls_report_seed_{seed}.txt'
    )
    
    print(pred_cell_type_composition_labels)

#### 2.1.6.1 Retrieve Embeddings

In [ ]:
# Cell embeddings
adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')

#### 2.1.6.2 Train Niche Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[niche_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[niche_key].value_counts()
print(cat_counts)
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[niche_key].isin(valid_cats)]

##### 2.1.6.2.1 Linear Classifier

In [ ]:
# Retrieve features and labels
niche_labels = pd.Categorical(adata_filtered.obs[niche_key]).codes
niche_categories = pd.Categorical(adata_filtered.obs[niche_key]).categories
niche_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = niche_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    niche_labels_train = niche_labels[train_indices]
    niche_labels_val = niche_labels[val_indices]
    niche_labels_test = niche_labels[test_indices]

    # Train and evaluate linear niche transfer classifier
    pred_niche_labels_test, _, _, model = linear_classifier(
        features_train=niche_features_train,
        labels_train=niche_labels_train,
        features_val=niche_features_val,
        labels_val=niche_labels_val,
        features_test=niche_features_test,
        labels_test=niche_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )
    
    #print(niche_categories[pred_niche_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{niche_key}_pred_linear'] = niche_categories[pred_niche_labels_test]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_linear_classifer_predicted_niches_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_linear',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.6.2.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
niche_features_train = niche_features[train_indices]
niche_features_test = niche_features[test_indices]
niche_labels_train = niche_labels[train_indices]
niche_labels_test = niche_labels[test_indices]

In [ ]:
# Train and evaluate knn niche transfer classifier
pred_niche_labels_test_knn = knn_classifier(
    features_train=niche_features_train,
    labels_train=niche_labels_train,
    features_test=niche_features_test,
    labels_test=niche_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_niche_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{niche_key}_pred_knn'] = niche_categories[pred_niche_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_niche_label_transfer_knn_classifer_predicted_niches.svg'
sc.pl.spatial(
    adata_test,
    color=f'{niche_key}_pred_knn',
    palette=niche_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.6.3 Train Cell Type Transfer Classifiers

In [ ]:
# Filter cells with missing labels
adata_filtered = adata_emb[~adata_emb.obs[cell_type_key].isna()]

# Filter cells with categories less than 'min_cat_count_thresh_values'
min_cat_count_thresh_values = 2
cat_counts = adata_filtered.obs[cell_type_key].value_counts()
valid_cats = cat_counts[cat_counts >= 2].index
adata_filtered = adata_filtered[adata_filtered.obs[cell_type_key].isin(valid_cats)]

##### 2.1.6.3.1 Linear Classifier

In [ ]:
# Retrieve features and labels
cell_type_labels = pd.Categorical(adata_filtered.obs[cell_type_key]).codes
cell_type_categories = pd.Categorical(adata_filtered.obs[cell_type_key]).categories
cell_type_features = adata_filtered.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get train and validation indices
    train_val_indices = (adata_filtered.obs['sample'] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_filtered.obs['sample'] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    cell_type_features_train = cell_type_features[train_indices]
    cell_type_features_val = cell_type_features[val_indices]
    cell_type_features_test = cell_type_features[test_indices]
    cell_type_labels_train = cell_type_labels[train_indices]
    cell_type_labels_val = cell_type_labels[val_indices]
    cell_type_labels_test = cell_type_labels[test_indices]

    # Train and evaluate linear cell type transfer classifier
    pred_cell_type_labels_test, _, _, model = linear_classifier(
        features_train=cell_type_features_train,
        labels_train=cell_type_labels_train,
        features_val=cell_type_features_val,
        labels_val=cell_type_labels_val,
        features_test=cell_type_features_test,
        labels_test=cell_type_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_linear_classifier_cls_report_seed_{seed}.txt'
        )

    #print(cell_type_categories[pred_cell_type_labels_test])

In [ ]:
# Store predictions from last run
adata_test = adata_list[1]
adata_test.obs[f'{cell_type_key}_pred_linear'] = cell_type_categories[pred_cell_type_labels_test]

In [ ]:
# Visualize predicted cell type labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_linear_classifer_predicted_cell_types_seed_{seed}.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_linear',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

##### 2.1.6.3.2 KNN Classifier

In [ ]:
# Get train and test indices
train_val_indices = (adata_filtered.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
test_indices = (adata_filtered.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]

# Split features and labels
cell_type_features_train = cell_type_features[train_indices]
cell_type_features_test = cell_type_features[test_indices]
cell_type_labels_train = cell_type_labels[train_indices]
cell_type_labels_test = cell_type_labels[test_indices]

In [ ]:
# Train and evaluate knn cell type transfer classifier
pred_cell_type_labels_test_knn = knn_classifier(
    features_train=cell_type_features_train,
    labels_train=cell_type_labels_train,
    features_test=cell_type_features_test,
    labels_test=cell_type_labels_test,
    results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_cell_type_label_transfer_knn_cls_report.txt')

In [ ]:
# Store predictions
adata_test.obs[f'{cell_type_key}_pred_knn'] = cell_type_categories[pred_cell_type_labels_test_knn]

In [ ]:
# Visualize predicted niche labels in test sample
sample = list(set(adata_test.obs[sample_key].tolist()))[0]
batch = list(set(adata_test.obs[batch_key].tolist()))[0]
print(sample)
file_name = f'{dataset_name}_{batch}_{method}_cell_type_label_transfer_knn_classifer_predicted_cell_types.svg'
sc.pl.spatial(
    adata_test,
    color=f'{cell_type_key}_pred_knn',
    palette=cell_type_palette,
    spot_size=spot_size,
    save=file_name)   

#### 2.1.6.4 Train Neighborhood Composition Classifiers

In [ ]:
# Compute spatial neighborhood graph
adata_batch_list = []
batches = adata_emb.obs[batch_key].unique().tolist()

for batch in batches:
    adata_batch = adata_emb[adata_emb.obs[batch_key] == batch]
    print('Computing spatial neighborhood...')
    # Construct neighbor graph
    adata_batch = construct_neighbor_graph(
        adata_batch,
        n_neighs=n_neighs,
        radius=None,
        delaunay=None,
        include_self_loop=False,
        compute_neighbor_counts=False)
    adata_batch_list.append(adata_batch)

adata_combined = sc.concat(adata_batch_list, join="inner")

# Combine spatial neighborhood graphs as disconnected components
batch_connectivities = []
len_before_batch = 0
for i in range(len(adata_batch_list)):
    if i == 0: # first batch
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[0].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[0].shape[0])))
        batch_connectivities.append(sp.hstack(
            (adata_batch_list[0].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    elif i == (len(adata_batch_list) - 1): # last batch
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0])))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'])))
    else: # middle batches
        before_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0], len_before_batch))
        after_batch_connectivities_extension = sp.csr_matrix(
            (adata_batch_list[i].shape[0],
            (adata_combined.shape[0] -
            adata_batch_list[i].shape[0] -
            len_before_batch)))
        batch_connectivities.append(sp.hstack(
            (before_batch_connectivities_extension,
            adata_batch_list[i].obsp['spatial_connectivities'],
            after_batch_connectivities_extension)))
    len_before_batch += adata_batch_list[i].shape[0]
connectivities = sp.vstack(batch_connectivities)
adata_combined.obsp['spatial_connectivities'] = connectivities

composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
cell_type_composition_labels = composition_df.values
niche_features = adata_emb.obsm[emb_key]

In [ ]:
n_seeds = 8

for seed in range(n_seeds):
    # Get neighborhood composition labels and features
    composition_df = compute_neighborhood_composition(adata_combined, cell_type_key)
    cell_type_composition_labels = composition_df.values
    niche_features = adata_emb.obsm[emb_key]
    
    # Get train and validation indices
    train_val_indices = (adata_emb.obs[sample_key] != 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    train_val_labels = cell_type_composition_labels[train_val_indices]
    train_indices, val_indices = train_test_split(
        train_val_indices,
        test_size=0.1,
        random_state=seed,
        #stratify=train_val_labels
    )
    
    # Get test indices
    test_indices = (adata_emb.obs[sample_key] == 'BK39_Lesional Baseline').to_numpy().nonzero()[0]
    
    # Split features and labels
    niche_features_train = niche_features[train_indices]
    niche_features_val = niche_features[val_indices]
    niche_features_test = niche_features[test_indices]
    cell_type_composition_labels_train = cell_type_composition_labels[train_indices]
    cell_type_composition_labels_val = cell_type_composition_labels[val_indices]
    cell_type_composition_labels_test = cell_type_composition_labels[test_indices]
    
    pred_cell_type_composition_labels, model = linear_regressor(
        features_train=niche_features_train,
        labels_train=cell_type_composition_labels_train,
        features_val=niche_features_val,
        labels_val=cell_type_composition_labels_val,
        features_test=niche_features_test,
        labels_test=cell_type_composition_labels_test,
        n_epochs=500,
        batch_size=128,
        results_save_path=f'{benchmarking_artifacts_folder_path}/results/{method}_{dataset_name}_neighborhood_composition_linear_classifier_cls_report_seed_{seed}.txt'
    )
    
    print(pred_cell_type_composition_labels)

In [ ]:
# Retrieve F1 scores
weighted_avg_f1_scores = []
for seed in range(n_seeds):
    with open(f'{benchmarking_artifacts_folder_path}/results/{dataset_name}_niche_label_transfer_cls_report_seed_{seed}.txt', "r") as file:
        for line in file:
            if line.strip().startswith("weighted avg"):
                # Extract all decimal numbers from the line
                numbers = re.findall(r"\d+\.\d+", line)
                if len(numbers) >= 3:
                    weighted_avg_f1_score = float(numbers[2])
                    print(f"Weighted Avg F1-Score: {weighted_avg_f1_score}")
                    weighted_avg_f1_scores.append(weighted_avg_f1_score)
print(np.mean(weighted_avg_f1_scores))

In [ ]:
method = 'scGPT-spatial'
for dataset_name in [
    #'xhs1000-39b_1p',
    #'chl59-8b_1p']:
    adata_emb = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded_clustered.h5ad')
    adata_cellplm = sc.read_h5ad(f'{benchmarking_artifacts_folder_path}/CellPLM/{dataset_name}_embedded.h5ad')
    adata_emb.obs = adata_cellplm.obs
    adata_emb.write(f'{benchmarking_artifacts_folder_path}/{method}/{dataset_name}_embedded.h5ad')
    